## Interior point method

In [1]:
import numpy as np
import pandas as pd
import copy as copy
import scipy.io
import time
import os
from scipy.linalg import solve, LinAlgWarning
import warnings
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import openpyxl
import xlsxwriter

from inpoint_methods import paso_intpoint, loadProblem, intpoint, intpointR, highlight_greaterthan #,intpointR_mask

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [2]:
def solve_catch(K,ld):
    with warnings.catch_warnings(record=True) as warneo:
        warnings.simplefilter("always", LinAlgWarning)
        # Solve the linear system
        w_vector = scipy.linalg.solve(K, ld)
        
        # Check if any warning was triggered
        if any(issubclass(warn.category, LinAlgWarning) for warn in warneo):
            # Print the warning message for all warnings captured
            for warn in warneo:
                print(f"Warning: {warn.message}")
    return w_vector

def highlight_df(mu, z, Q, k, tau, highlighted_df, mudf, epsilon=1e-6, comp_tol=1e-5):
    # Previous μ from mudf
    prev_mu = mudf.loc[k-1].values
    
    # Only consider indices where μ*z <= comp_tol
    mask = mu * z <= comp_tol
    
    # Highlight if μ did not increase significantly and μ*z is small enough
    highlighted_rows = [i for i in range(len(mu)) if mask[i] and mu[i] <= prev_mu[i] + epsilon]
    
    # Build row of 1s and 0s
    highlighted_row = [1 if i in highlighted_rows else 0 for i in range(Q.shape[0])]
    
    # Store in DataFrame
    highlighted_df.loc[k] = highlighted_row
    return highlighted_df

def display_highlights(highlighted_df,regression_df):
    # Take last 3 iterations
    last_3_iterations = highlighted_df.tail(3)
    
    # Columns consistently highlighted (1 in all last 3 iterations)
    highlighted_columns = last_3_iterations.columns[(last_3_iterations == 1).all(axis=0)].tolist()
    highlighted_columns = [int(x) for x in highlighted_columns]  # ensure ints
    print("Consistently highlighted in last 3 iterations:", highlighted_columns)
    print("How many?=", len(highlighted_columns))
    
    # Columns that were highlighted in previous iteration but regressed this iteration (1 → 0)
    regressed_columns = []
    if len(highlighted_df) >= 2:
        prev_iter = highlighted_df.iloc[-2]
        last_iter = highlighted_df.iloc[-1]
        regressed_columns = prev_iter.index[(prev_iter == 1) & (last_iter == 0)].tolist()
        regressed_columns = [int(x) for x in regressed_columns]
        if regressed_columns:
            print(f"WARNING: Iter {highlighted_df.index[-1]} regressed columns (1 -> 0):", regressed_columns)

            highlighted_df_regressed = highlighted_df.tail(10)[regressed_columns]
            display(highlighted_df_regressed)
        iter_idx = highlighted_df.index[-1]
        if regression_df is not None:
                row = pd.Series(0, index=highlighted_df.columns)
                row[regressed_columns] = 1
                regression_df.loc[iter_idx] = row
    
    return highlighted_columns, regressed_columns


def progress_summary_df_clean(v2, v3, v4, tau, G, x, Q, c, ld1,
                                 main_norm, pr_before, inq_before, cmp_before,
                                 tau_before, cond_before, obj_before):
    """
    Return a DataFrame with the progress summary and an 'Improved?' column.
    Marks True if the metric improved after the step.
    """

    # Compute "after" values
    pr_after   = np.linalg.norm(v2, np.inf)
    inq_after  = np.linalg.norm(v3, np.inf)
    cmp_after  = np.max(v4)
    tau_after  = tau
    cond_after = np.linalg.cond(G, 1)
    obj_after  = 0.5 * x @ Q @ x + c @ x
    norm_after = np.linalg.norm(ld1, np.inf)

    # Define metrics, before and after
    metrics = [
        ("overall ||ld||∞", main_norm, norm_after),
        ("primal ||·||∞", pr_before, pr_after),
        ("ineq ||·||∞", inq_before, inq_after),
        ("max(mu*z)", cmp_before, cmp_after),
        ("tau", tau_before, tau_after),
        ("cond(G)", cond_before, cond_after),
        ("objective", obj_before, obj_after)
    ]

    # Build table with improvement
    summary_data = {
        "Metric": [],
        "Before": [],
        "After": [],
        "Improved?": []
    }

    for name, before, after in metrics:
        summary_data["Metric"].append(name)
        summary_data["Before"].append(before)
        summary_data["After"].append(after)
        
        # Determine improvement
        # Smaller is better for residuals, complementarity, tau, cond(G)
        # For objective, smaller is better if minimizing
        if name in ["overall ||ld||∞", "primal ||·||∞", "ineq ||·||∞", "max(mu*z)", "tau", "cond(G)", "objective"]:
            improved = after < before
        else:
            improved = np.nan  # unknown metric

        summary_data["Improved?"].append(improved)

    summary_df = pd.DataFrame(summary_data)

    return summary_df, pr_after, inq_after, cmp_after, tau_after, cond_after, obj_after, norm_after

def build_reduced_system(Q, AT, FT, U, A, F, Z, mu, x, lamda, c, b, d, tau, highlighted_columns):
    # Build block rows
    m = A.shape[0]
    p = U.shape[0]
    n = Q.shape[0]

    r1 = np.hstack((Q, AT, -FT @ U))
    r2 = np.hstack((A, np.zeros((m, m + p))))
    r3 = np.hstack((-U @ F, np.zeros((p, m)), -Z @ U))

    M1 = np.vstack((r1, r2, r3))

    # Step 1: Filter mu
    mu_filtered = np.array([mu[i] for i in range(len(mu)) if i not in highlighted_columns])

    # Step 2: Mapping matrix (positions old -> new)
    p_filtered = len(mu_filtered)
    mapping_matrix = np.zeros((p, p_filtered), dtype=int)
    for new_pos, old_pos in enumerate([i for i in range(p) if i not in highlighted_columns]):
        mapping_matrix[old_pos, new_pos] = 1

    # Step 3: Build U1
    U1 = np.diag(mu_filtered)

    # Step 4: Remove rows and columns
    rows_to_remove = [i + (n + m) for i in highlighted_columns]
    num_removed = len(rows_to_remove)
    M1 = np.delete(M1, rows_to_remove, axis=0)
    M1 = np.delete(M1, rows_to_remove, axis=1)

    print(f"Deleted {num_removed} rows/columns. New M1 shape: {M1.shape}")

    if M1.shape[0] != M1.shape[1]:
        raise ValueError("M1 is not square! Check highlighted indices.")

    # Step 5: Build reduced RHS vector
    ld1 = np.concatenate((
        Q @ x + AT @ lamda - FT @ mu + c,
        A @ x - b,
        U @ (d - F @ x) + tau
    ))
    ld1 = np.delete(ld1, rows_to_remove, axis=0)

    return M1, U1, ld1

def load_lp_problem(mat_file: str):
    """
    Loads a linear/quadratic problem from a .mat file and initializes
    the problem matrices for the interior-point algorithm.
    
    Returns:
        Q : ndarray    Quadratic matrix (identity by default)
        c : ndarray    Linear term vector
        A : ndarray    Equality constraint matrix
        b : ndarray    Equality constraint RHS
        F : ndarray    Inequality constraint matrix (identity by default)
        d : ndarray    Inequality constraint RHS (zeros)
        H : dict       Raw data loaded from the .mat file
    """
    print(f"Loading problem from: {mat_file}")
    H = loadProblem(f"mat_files/{mat_file}")

    # Quadratic term: identity (can be changed if needed)
    Q = np.eye(H['c'].shape[0])

    # Linear term
    c = H['c'].ravel()  # flatten in case it's a column vector

    # Equality constraints
    A = H['AE']
    b = H['bE'].ravel()  # flatten in case it's a column vector

    # Compute percentage of zeros in b
    num_zeros_b = np.sum(b == 0)
    pct_zeros_b = 100 * num_zeros_b / len(b)
    
    # Inequality constraints (x >= 0 by default)
    F = np.eye(H['c'].shape[0])
    d = np.zeros(H['c'].shape[0])

    print(f"Problem loaded. n={Q.shape[0]}, m={A.shape[0]}, p={F.shape[0]}")
    print(f"Equality RHS b: {b}")
    print(f"Number of zeros in b: {num_zeros_b} ({pct_zeros_b:.2f}%)")

    return Q, c, A, b, F, d, H


Load problem

In [3]:
#mat_files = 'lp_kb2.mat'     # b = 0
#mat_files = 'lp_afiro.mat'   # bmax = 500, and no mu tends to zero  and no condition problem
mat_files = 'lp_blend.mat'    # bmax = 26.32, and no mu tends to zero.
#mat_files = 'lp_fit1d.mat'   # b = 0
#mat_files = 'lp_fit1p.mat'   # bmax = 216, and no mu tends to zero, condition 10^12
#mat_files = 'lp_grow15.mat'  # b = 0, condition 10^6
#mat_files = 'lp_grow22.mat'  # b = 0, condition 1.4e7
#mat_files = 'lp_grow7.mat'   # b = 0, condition 1.2e7

print(mat_files)

H = loadProblem(f"mat_files/{mat_files}")

# Define the quadratic matrix Q
Q = np.eye(H['c'].shape[0])

# Define the linear term vector c
c = H['c']

# Define the equality constraint matrix A and vector b
A = H['AE']
b = H['bE']
#b = np.zeros( A.shape[0] )

# Define an inequality constraint: x1 >= -10
#F = np.zeros([H['c'].shape[0],H['c'].shape[0]])
F = np.eye(H['c'].shape[0])
d = np.zeros([H['c'].shape[0],1])
d = d.ravel()  # Flattens the vector to a 1D vector
print(b)

lp_blend.mat
 Norma infinita de b:  26.32
[ 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.   23.26  5.25 26.32 21.05 13.45  2.58 10.   10.
  0.    0.  ]


Sistema Completo

In [4]:
#x, lamda, mu, z, k = intpoint(Q, A, F, c, b, d)

Sistema Reducido

In [5]:
#x, lamda, mu, z, k = intpointR(Q, A, F, c, b, d)

# Now let's do it manually

So let's say we got to the end of that iteration and I already know which ones I wanna use

In [5]:
# Choose a file
#mat_files = 'lp_kb2.mat'     # b = 0
#mat_files = 'lp_afiro.mat'   # bmax = 500, and no mu tends to zero  and no condition problem
#mat_files = 'lp_blend.mat'    # bmax = 26.32, and no mu tends to zero.
mat_files = 'lp_fit1d.mat'   # b = 0
#mat_files = 'lp_fit1p.mat'   # bmax = 216, and no mu tends to zero, condition 10^12
#mat_files = 'lp_grow15.mat'  # b = 0, condition 10^6
#mat_files = 'lp_grow22.mat'  # b = 0, condition 1.4e7
#mat_files = 'lp_grow7.mat'   # b = 0, condition 1.2e7

# Load problem
Q, c, A, b, F, d, H = load_lp_problem(mat_files)

k = 0
n = Q.shape[0]
#print("n= ",n, "Q.shape[0]")
m = A.shape[0]
#print("m= ",m, "A.shape[0]")
p = F.shape[0]
#print("p= ",p, "F.shape[0]")

problem_info = {
    "problem_name": mat_files,
    "norm_inf_b": np.linalg.norm(b, np.inf),
    "dim_b": len(b),
    "num_zeros_b": np.sum(b==0),
    "dim_x": n,
    "dim_eq": m,
    "dim_ineq": p
}

tol = 1e-6
kmax = 10

#print("El rango de A es", np.linalg.matrix_rank(A))

AT = A.T
FT = F.T

# Initial values
x = np.ones(n)
lamda = np.zeros(m)
mu = np.ones(p)
z = np.ones(p)

sigma = 0.5 # Valor fijo
tau = sigma * np.dot(mu, z) / p

# Dataframes that store mu and z values on every iteration.
# Each iteration is a new row added after this.
# Used for the graphs
mudf = pd.DataFrame(columns=range(p))
zdf = pd.DataFrame(columns=range(p))
taudf = pd.DataFrame(columns=range(p))
ratiodf = pd.DataFrame(columns=range(p))
highlighted_df = pd.DataFrame(columns=range(p))

#Dataframes which record mu, z and tau over each iteration
mudf.loc[k] = mu
zdf.loc[k] = z
taudf.loc[k] = np.full(p, tau)
ratiodf.loc[k] = mu / tau

v1 = Q @ x + AT @ lamda - FT @ mu + c
v2 = A @ x - b
v3 = -F @ x + d + z
v4 = np.multiply(mu, z)  # Element-wise product
ld1 = np.concatenate((v1, v2, v3, v4), 0)
norma_cnpo = np.linalg.norm(ld1,np.inf) # this is the CNPO without the perturbation
w = np.zeros((p, 1))

# Initialize an empty DataFrame to store the iteration results
highlighted_df = pd.DataFrame(columns=range(p))  # drop/test inequalities → p columns
regression_df = pd.DataFrame(columns=highlighted_df.columns) #store any regressed indexes for mu

red_mu = []

while norma_cnpo > tol and k < kmax:
    # Update diagonal matrices Z and U inside the loop
    # Initial residuals
    Z = np.diag(z)
    U = np.diag(mu)
    ### KKT Matrix
    row1 = np.hstack((Q, AT, -FT, np.zeros((n, p))))
    row2 = np.hstack((A, np.zeros((m, m + p + p))))
    row3 = np.hstack((-F, np.zeros((p, m + p)), np.identity(p)))
    row4 = np.hstack((np.zeros((p, n + m)), Z, U))

    M = np.vstack((row1, row2, row3, row4))
    
    D = np.diag(mu / z)
    G = Q+FT@D@F
    for i in range(p):
        w[i] = F[i, :] @ x - d[i] - (tau / mu[i])
    w = w.ravel()
        
    dg = Q @ x + AT @ lamda - FT@mu + c + FT@D@w
    
    # Define K as a block matrix
    m = A.shape[0]
    K = np.block([
        [G, AT],
        [A, np.zeros((m, m))]
    ])
    
    # Calculate the reciprocal condition number of G
    condK = np.linalg.cond(G,1)
    
    # Define ld
    ld = -np.concatenate([dg, A @ x - b])
    #norma_cnpo = np.linalg.norm(ld, np.inf)
    
    # Solve the linear system and catch errors

    w_vector = solve_catch(K,ld) # Reduced system
    
    # Update the sections of the w_vector
    wx     = w_vector[0:n]
    wlamda = w_vector[n:n + m]
    wmu    = - D @ (F @ wx + w)
    wz     = - ( (1 / mu) * (z * wmu - tau) + z )
    
    ### Step size
    alpha_mu = paso_intpoint(mu, wmu)
    alpha_z  = paso_intpoint(z, wz)
    #alpha    = 0.995 * min(alpha_mu, alpha_z)
    alpha    = min(alpha_mu, alpha_z)
    #print("alpha= " , alpha)
    
    # remember something
    perc_mu = wmu/mu
    perc_z  = wz/z
    
    # Update variables
    x += alpha * wx
    mu += alpha * wmu
    lamda += alpha * wlamda
    z += alpha * wz
    
    # Update tau and residuals
    tau = sigma * np.dot(mu, z) / (p)
    k += 1

    mudf.loc[k] = mu    # dataframe for graphing the central path
    zdf.loc[k] = z  
    taudf.loc[k] = np.full(p, tau)     # dataframe for graphing the central path
    ratiodf.loc[k] = mu / tau

    highlighted_df = highlight_df(mu, z, Q, k, tau, highlighted_df, mudf)

    # Recalculate residuals
    v1 = Q @ x + AT @ lamda - FT @ mu + c
    v2 = A @ x - b
    v3 = -F @ x + d + z
    v4 = np.multiply(mu, z)  # Element-wise product
    
    ld1 = np.concatenate((v1, v2, v3, v4), 0)
    norma_cnpo = np.linalg.norm(ld1,np.inf)

    main_norm = norma_cnpo # CHECK IF THIS IS CORRECT HERE
    pr_before   = np.linalg.norm(v2, np.inf) # before name is misleading
    inq_before  = np.linalg.norm(v3, np.inf) # before name is misleading
    cmp_before  = np.max(v4) # before name is misleading
    tau_before  = tau # before name is misleading
    cond_before = condK # before name is misleading
    obj_before  = 0.5 * x @ Q @ x + c @ x # before name is misleading
    
    #print("\niter=", k, "\t", "||cnpo||=", norma_cnpo)
    #print("Condition number of G:", np.linalg.cond(G,1))
    #print("rcond(G)", (1/np.linalg.cond(G,1)))
    #print("tau =",tau)
    #print(z)
    #print(mu)
    
    mask = mu*z <= 1e-5
    
    #print('cuantos chicos mu*z = %g, vector\n' % (sum(mask)), (mu*z)[mask])

    if all(mask): #Return True only if every entry in mask is True
        # Once the mu and z have gotten sufficiently small,
        # we record where we were before we start with the heuristic/experiment:

        #neg_mu_mask = (-0.52 < perc_mu) & (perc_mu < -0.48)
        #const_z_mask = (-0.01 < perc_z) & (perc_z < 0.01)
        grow_z_mask = (perc_z > -0.03) #& (perc_z < 0.01)
        neg_mu = np.arange( len(mask) )[grow_z_mask]
        
        highlighted_df = highlight_df(mu, z, Q, k, tau, highlighted_df, mudf)
        
        #print('mus chicos: vector\n', mu[neg_mu])
        if set(red_mu).issubset( neg_mu ):
            print ('IS subset: GOOD')
        else:
            print ('FAILS subset condition: BAD')
            
        #print('mus chicos: vector\n', neg_mu)
        #print('  change in percentages for mu \n', perc_mu[neg_mu] )
        #print('zs tending to positive contants\n', z[neg_mu] )
        #print('  Largest and smallest change for percentages in entries of z  \n', min(perc_z[neg_mu]), max(perc_z[neg_mu] ))
        red_mu = neg_mu.copy()
    
    # So here is when we start getting close to the issues of the matrix, as we already exited the while loop

k_red = 0
tol_red = 1e-10
max_red = 10
#print("CHECK BEFORE REDUCTION")
highlighted_columns, regressed_columns = display_highlights(highlighted_df,regression_df)
regressed_df = pd.DataFrame(columns=['Iteration', 'Regressed_columns'])

while k_red < max_red:
#while np.linalg.norm(ld1, np.inf) > tol_red and k_red < max_red:
    Z = np.diag(z)
    U = np.diag(mu)

    M1, U1, ld1 = build_reduced_system(Q, AT, FT, U, A, F, Z, mu, x, lamda, c, b, d, tau, highlighted_columns)

    print("cond(M1, 1-norm):", np.linalg.cond(M1, 1))

    # SOLVE THE NEW SYSTEM
    w_vector1 = scipy.linalg.solve(M1, ld1)

    wx1     = w_vector1[:n] # should be the same ?
    wlamda1 = w_vector1[n:n + m] # should be the same ?
    Dwmu1    = w_vector1[n + m:]
    
    wmu1 = U1 @ Dwmu1  # Divide each element of wmu1 by the corresponding diagonal element of U

    # Here's the vector transformed back into the original size problem
    full_wmu = np.zeros_like(mu)
    active_indices = [i for i in range(p) if i not in highlighted_columns]
    for i, idx in enumerate(active_indices):
        full_wmu[idx] = wmu1[i]

    # ────────── complete here ──────────
    # 1. Reconstruct Δz from the 3rd KKT row (primal feasibility)
    # δz = F δx - (Fx - d - z)  == F @ wx1 - v3  (since v3 = -F@x + d + z)
    full_wz = F @ wx1 - (F @ x - d - z)

    # Recompute step sizes for the reduced-step directions **DOUBLE CHECK THIS LATER
    alpha_mu = paso_intpoint(mu, full_wmu)   # ensures μ + α·Δμ ≥ (1-τ)μ > 0
    alpha_z  = paso_intpoint(z,  full_wz)    # ensures z + α·Δz ≥ (1-τ)z > 0
    alpha    = 0.995 * min(alpha_mu, alpha_z)

    # Compute perc_mu, perc_z for the reduced step
    perc_mu = full_wmu / mu
    perc_z  = full_wz / z

    # 2. Apply the step with the same step‑size alpha
    x      += alpha * wx1
    lamda  += alpha * wlamda1
    mu     += alpha * full_wmu
    z      += alpha * full_wz

    # 4. Refresh τ and bump iteration counter (optional, for bookkeeping)
    tau = sigma * np.dot(mu, z) / p

    #print("max change in x:", np.max(np.abs(alpha*wx1)))
    #print("max change in mu:", np.max(np.abs(alpha*full_wmu)))
    #print("max change in z:", np.max(np.abs(alpha*full_wz)))

    # 3. Recompute residuals for diagnostics
    v1 = Q @ x + AT @ lamda - FT @ mu + c      # dual residual
    v2 = A @ x - b                             # primal residual
    v3 = -F @ x + d + z                        # inequality residual
    v4 = mu * z                                # complementarity product
    ld1 = np.concatenate((v1, v2, v3, v4))

    norm_after = np.linalg.norm(ld1, np.inf)

    k_red += 1
    total_iter= k + k_red
    #print("total_iter:", total_iter)

    # Update highlighted_df after this reduced iteration

    mudf.loc[total_iter] = mu    # dataframe for graphing the central path
    zdf.loc[total_iter] = z  
    taudf.loc[total_iter] = np.full(p, tau)     # dataframe for graphing the central path
    ratiodf.loc[total_iter] = mu / tau
    
    highlighted_df = highlight_df(mu, z, Q, total_iter, tau, highlighted_df, mudf)

    # Optionally, recompute the highlighted columns for the next reduced iteration
    highlighted_columns, regressed_columns = display_highlights(highlighted_df,regression_df)

    # AFTER computing highlighted_columns, regressed_columns in the reduced loop:

    if len(regressed_columns) > 0:
        regressed_df = pd.concat([regressed_df,
                                pd.DataFrame({'Iteration': [k_red],
                                                'Regressed_columns': [regressed_columns]})],
                                ignore_index=True)
        print(f"Iteration of reduced loop {k_red}: Regressed columns:", regressed_columns)
    else:
        print(f"Iteration of reduced loop  {k_red}: No regressed columns")

summary_df, pr_after, inq_after, cmp_after, tau_after, cond_after, obj_after, norm_after = \
        progress_summary_df_clean(v2, v3, v4, tau, G, x, Q, c, ld1,
                                main_norm, pr_before, inq_before, cmp_before,
                                tau_before, cond_before, obj_before)

display(summary_df)

# Save to excel file
# After finishing the loops
# After your existing computation and summary_df creation:

# Add problem info
summary_df['Problem'] = mat_files
summary_df['dim_x'] = n
summary_df['dim_A'] = m
summary_df['dim_F'] = p
summary_df['||b||_inf'] = np.linalg.norm(b, np.inf)
summary_df['num_zeros_in_b'] = np.sum(b == 0)

# Reorder columns if you want
cols_order = ['Problem', 'dim_x', 'dim_A', 'dim_F', '||b||_inf', 'num_zeros_in_b', 'Metric', 'Before', 'After']
summary_df = summary_df[cols_order]

# After finishing all computations:

# After all your iterations and building summary_df and regressed_df

with pd.ExcelWriter(f"progress_summary_{mat_files}.xlsx", engine='xlsxwriter') as writer:
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    if 'regressed_df' in locals() and not regressed_df.empty:
        regressed_df.to_excel(writer, sheet_name='Regressions', index=False)

print(f"Progress summary with regressions saved for {mat_files}")

print("Condition number (1-norm):", cond_after)
print("Objective after:", obj_after)

Loading problem from: lp_fit1d.mat
 Norma infinita de b:  0.0
Problem loaded. n=1049, m=24, p=1049
Equality RHS b: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Number of zeros in b: 24 (100.00%)
Consistently highlighted in last 3 iterations: []
How many?= 0
Deleted 0 rows/columns. New M1 shape: (2122, 2122)
cond(M1, 1-norm): 6302798.411176298
Consistently highlighted in last 3 iterations: []
How many?= 0
Iteration of reduced loop  1: No regressed columns
Deleted 0 rows/columns. New M1 shape: (2122, 2122)
cond(M1, 1-norm): 236509211.8602612
Consistently highlighted in last 3 iterations: []
How many?= 0
Iteration of reduced loop  2: No regressed columns
Deleted 0 rows/columns. New M1 shape: (2122, 2122)
cond(M1, 1-norm): 308559229.4571275
Consistently highlighted in last 3 iterations: []
How many?= 0
Iteration of reduced loop  3: No regressed columns
Deleted 0 rows/columns. New M1 shape: (2122, 2122)
cond(M1, 1-norm): 309647661.3761182
Consistently highlighte

,1035
6,0
7,0
8,0
9,0
10,0
11,0
12,1
13,1
14,1
15,0


Iteration of reduced loop 5: Regressed columns: [1035]
Deleted 0 rows/columns. New M1 shape: (2122, 2122)
cond(M1, 1-norm): 648819737.66578
Consistently highlighted in last 3 iterations: []
How many?= 0
Iteration of reduced loop  6: No regressed columns
Deleted 0 rows/columns. New M1 shape: (2122, 2122)
cond(M1, 1-norm): 32785414558.218372
Consistently highlighted in last 3 iterations: []
How many?= 0
Iteration of reduced loop  7: No regressed columns
Deleted 0 rows/columns. New M1 shape: (2122, 2122)
cond(M1, 1-norm): 2172662672275.607
Consistently highlighted in last 3 iterations: [1037]
How many?= 1
Iteration of reduced loop  8: No regressed columns
Deleted 1 rows/columns. New M1 shape: (2121, 2121)
cond(M1, 1-norm): 50405279.2164822
Consistently highlighted in last 3 iterations: [1037]
How many?= 1
Iteration of reduced loop  9: No regressed columns
Deleted 1 rows/columns. New M1 shape: (2121, 2121)
cond(M1, 1-norm): 1799233483.6164715
Consistently highlighted in last 3 iterations: 

,Metric,Before,After,Improved?
0,overall ||ld||∞,4.590603e+04,5.101470e+04,False
1,primal ||·||∞,4.590603e+04,5.101470e+04,False
2,ineq ||·||∞,1.776357e-15,3.552714e-15,False
3,max(mu*z),8.426863e-01,9.173820e-01,False
4,tau,1.807406e-01,1.776399e-01,True
5,cond(G),1.236521e+06,1.236521e+06,False
6,objective,-3.279245e+04,-1.927656e+04,False


Progress summary with regressions saved for lp_fit1d.mat
Condition number (1-norm): 1236520.999055649
Objective after: -19276.55906303805


# Observing results

Ok so we're at the point right now in which I have finally finished coding the actual stuff I had on my paper, but now I need to observe the results. What are we actually seeing here? So the things we want to check out are the:
- Matrix condition
- Central path graph
- Proper b vector
- Observe the decreases in the $mu$

## 1. Central path

So I'm gonna make a small provisional code in which I can observe the central path for each $mu_i$ and $z_i$
For kb2, the dim(mu)=dim(z)=p=68
So that's a shitton of graphs. I'm gonna sample 10 values out of 68 of their indexes and register those values during the iterations

So there's three things to do here
1. Save the mu values for each iteration in a dataframe, each iteration is a row
2. Sample 10 mu indexes
3. Graph them at the end of the method.

Let's remember how to do all of that HAHAHAHA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

np.random.seed(42)
columns = np.random.choice(mudf.columns, size=min(10, len(mudf.columns)), replace=False)

for col in columns:
    fig, ax = plt.subplots()
    scat = ax.plot([], [], 'o', markersize=4, label=f'col {col}')[0]
    ax.set_xlim(mudf[col].min() - 0.1, mudf[col].max() + 1)
    ax.set_ylim(zdf[col].min() - 0.1, zdf[col].max() + 1)
    ax.set_xlabel('mu')
    ax.set_ylabel('z')
    ax.set_title(f'$\\mu_{{{col}}}$ vs. $z_{{{col}}}$ over iterations')
    ax.legend()

    def update(frame):
        x = mudf[col].iloc[:frame+1]
        y = zdf[col].iloc[:frame+1]
        scat.set_data(x, y)
        return scat,

    ani = animation.FuncAnimation(fig, update, frames=len(mudf), interval=100, blit=True)
    out_path = f'CPgifsbnotzero/mu_vs_z_col{col}.gif'
    ani.save(out_path, writer='pillow')
    print(f'Saved: {out_path}')
    plt.close()


Sistema completo

In [ ]:
k = 0
n = Q.shape[0]
#print("n= ",n, "Q.shape[0]")
m = A.shape[0]
#print("m= ",m, "A.shape[0]")
p = F.shape[0]
#print("p= ",p, "F.shape[0]")

tol = 1e-6
kmax = 100

#print("El rango de A es", np.linalg.matrix_rank(A))

AT = A.T
FT = F.T

# Initial values
x = np.ones(n)
lamda = np.zeros(m)
mu = np.ones(p)
z = np.ones(p)

sigma = 0.5 # Valor fijo
tau = sigma * np.dot(mu, z) / p

# Dataframes that store mu and z values on every iteration.
# Each iteration is a new row added after this.
# Used for the graphs
mudf = pd.DataFrame(columns=[f"{i}" for i in range(len(mu))])
zdf = pd.DataFrame(columns=[f"{i}" for i in range(len(z))])
mudf.loc[k] = mu
zdf.loc[k] = z

v1 = Q @ x + AT @ lamda - FT @ mu + c
v2 = A @ x - b
v3 = -F @ x + d + z
v4 = np.multiply(mu, z)  # Element-wise product
ld = np.concatenate((v1, v2, v3, v4), 0)
norma_cnpo = np.linalg.norm(ld,np.inf)
w = np.zeros((p, 1))

# Initialize an empty DataFrame to store the iteration results
highlighted_df = pd.DataFrame(columns=range(p))  # drop/test inequalities → p columns

while norma_cnpo > tol and k < kmax:
    # Update diagonal matrices Z and U inside the loop
    # Initial residuals
    Z = np.diag(z)
    U = np.diag(mu)
    v4_pert = np.multiply(mu, z) - tau * np.ones(p)

    ### KKT Matrix
    row1 = np.hstack((Q, AT, -FT, np.zeros((n, p))))
    row2 = np.hstack((A, np.zeros((m, m + p + p))))
    row3 = np.hstack((-F, np.zeros((p, m + p)), np.identity(p)))
    row4 = np.hstack((np.zeros((p, n + m)), Z, U))

    K = np.vstack((row1, row2, row3, row4))
    
    #D = np.diag(mu / z)
    #G = Q+FT@D@F
    #for i in range(p):
    #    w[i] = F[i, :] @ x - d[i] - (tau / mu[i])
    #w = w.ravel()
        
    #dg = Q @ x + AT @ lamda - FT@mu + c + FT@D@w
    
    # Define K as a block matrix
    #m = A.shape[0]
    #K = np.block([
    #    [G, AT],
    #    [A, np.zeros((m, m))]
    #])
    
    # Calculate the reciprocal condition number of G
    condK = np.linalg.cond(K,1)
    ld_pert = np.concatenate((v1, v2, v3, v4_pert), 0)
    
    # Define ld
    #ld = -np.concatenate([dg, A @ x - b])
    norma_cnpo = np.linalg.norm(ld_pert, np.inf)
    
    # Solve the linear system and catch errors
    
    w_vector = scipy.linalg.solve(K, -ld_pert) # Reduced system
    
    # Update the sections of the w_vector
    wx = w_vector[0:n]
    wlamda = w_vector[n:n + m]
    wmu = w_vector[n + m:n + m + p]
    wz = w_vector[n + m + p:]
    
    ### Step size
    alpha_mu = paso_intpoint(mu, wmu)
    alpha_z  = paso_intpoint(z, wz)
    #alpha    = 0.995 * min(alpha_mu, alpha_z)
    alpha    = min(alpha_mu, alpha_z)
    #print("alpha= " , alpha)
    
    # remember something
    perc_mu = wmu/mu
    perc_z  = wz/z
    
    # Update variables
    x += alpha * wx
    mu += alpha * wmu
    lamda += alpha * wlamda
    z += alpha * wz

    mudf.loc[k] = mu
    zdf.loc[k] = z
    
    # Update tau and residuals
    tau = sigma * np.dot(mu, z) / (p)
    k += 1
    
    # Recalculate residuals
    v1 = Q @ x + AT @ lamda - FT @ mu + c
    v2 = A @ x - b
    v3 = -F @ x + d + z
    v4 = np.multiply(mu, z)  # Element-wise product
    
    ld1 = np.concatenate((v1, v2, v3, v4), 0)
    norma_cnpo = np.linalg.norm(ld1,np.inf)
    
    #print("\niter=", k, "\t", "||cnpo||=", norma_cnpo)
    #print("Condition number of G:", np.linalg.cond(G,1))
    #print("rcond(G)", (1/np.linalg.cond(G,1)))
    #print("tau =",tau)
    #print(z)
    #print(mu)
    
    mask = mu*z <= 1e-5
    
    #print('cuantos chicos mu*z = %g, vector\n' % (sum(mask)), (mu*z)[mask])
    norm_before = norma_cnpo 

    red_mu = []
    if all(mask):
        pr_before   = np.linalg.norm(v2, np.inf)
        inq_before  = np.linalg.norm(v3, np.inf)
        cmp_before  = np.max(v4)
        tau_before  = tau
        cond_before = condK
        obj_before  = 0.5 * x @ Q @ x + c @ x
        norm_before = norma_cnpo

        #neg_mu_mask = (-0.52 < perc_mu) & (perc_mu < -0.48)
        #const_z_mask = (-0.01 < perc_z) & (perc_z < 0.01)
        grow_z_mask = (perc_z > -0.03) #& (perc_z < 0.01)
        neg_mu = np.arange( len(mask) )[grow_z_mask]
        
        highlighted_df = highlight_df(mu,perc_mu,z,perc_z,Q,k)
        
        #print('mus chicos: vector\n', mu[neg_mu])
        if set(red_mu).issubset( neg_mu ):
            print ('IS subset: GOOD')
        else:
            print ('FAILS subset condition: BAD')
            
        #print('mus chicos: vector\n', neg_mu)
        #print('  change in percentages for mu \n', perc_mu[neg_mu] )
        #print('zs tending to positive contants\n', z[neg_mu] )
        #print('  Largest and smallest change for percentages in entries of z  \n', min(perc_z[neg_mu]), max(perc_z[neg_mu] ))
        red_mu = neg_mu.copy()
    
    # So here is when we start getting close to the issues of the matrix
    if norma_cnpo <= tol or k == kmax:
        print(f"HELLO K={k}")
        k_red = 0
        tol_red = 1e-10
        max_red = 5

        while k_red < max_red:
        #while np.linalg.norm(ld1, np.inf) > tol_red and k_red < max_red:
            Z = np.diag(z)
            U = np.diag(mu)
            highlighted_columns, regressed_columns = display_highlights(highlighted_df)
            M1, U1, ld1 = build_reduced_system(Q, AT, FT, U, A, F, Z, mu, x, lamda, c, b, d, tau, highlighted_columns)

            print("cond(M1, 1-norm):", np.linalg.cond(M1, 1))

            # SOLVE THE NEW 3X3 SYSTEM
            w_vector1 = scipy.linalg.solve(M1, ld1)

            ### OK SO UP TILL NOW I JUST SOLVED ONE ITERATION OF THIS METHOD AND NOW I NEED TO OBSERVE THESE RESULTS
            ## NOW I NEED TO CHANGE THESE

            wx1     = w_vector1[:n] # should be the same ?
            wlamda1 = w_vector1[n:n + m] # should be the same ?
            Dwmu1    = w_vector1[n + m:]
            
            wmu1 = U1 @ Dwmu1  # Divide each element of wmu1 by the corresponding diagonal element of U

            # OK so now what? We need to analyse the problem and see if we're getting any closer.
            # Here's the vector transformed back into the original size problem
            full_wmu = np.zeros_like(mu)
            active_indices = [i for i in range(p) if i not in highlighted_columns]
            for i, idx in enumerate(active_indices):
                full_wmu[idx] = wmu1[i]

            # ────────── complete here ──────────
            # 1. Reconstruct Δz from the 3rd KKT row (primal feasibility)
            # δz = F δx - (Fx - d - z)  == F @ wx1 - v3  (since v3 = -F@x + d + z)
            full_wz = F @ wx1 - (F @ x - d - z)

            # Recompute step sizes for the reduced-step directions **DOUBLE CHECK THIS LATER
            alpha_mu = paso_intpoint(mu, full_wmu)   # ensures μ + α·Δμ ≥ (1-τ)μ > 0
            alpha_z  = paso_intpoint(z,  full_wz)    # ensures z + α·Δz ≥ (1-τ)z > 0
            alpha    = 0.995 * min(alpha_mu, alpha_z)

            # 2. Apply the step with the same step‑size alpha
            x      += alpha * wx1
            lamda  += alpha * wlamda1
            mu     += alpha * full_wmu
            z      += alpha * full_wz

            #print("max change in x:", np.max(np.abs(alpha*wx1)))
            #print("max change in mu:", np.max(np.abs(alpha*full_wmu)))
            #print("max change in z:", np.max(np.abs(alpha*full_wz)))

            # 3. Recompute residuals for diagnostics
            v1 = Q @ x + AT @ lamda - FT @ mu + c      # dual residual
            v2 = A @ x - b                             # primal residual
            v3 = -F @ x + d + z                        # inequality residual
            v4 = mu * z                                # complementarity product
            ld1 = np.concatenate((v1, v2, v3, v4))
            norm_after = np.linalg.norm(ld1, np.inf)

            # 4. Refresh τ and bump iteration counter (optional, for bookkeeping)
            tau = np.dot(mu, z) / (2 * p)
            k  += 1
            k_red += 1

            # Optional: save to Excel
            
        summary_df, pr_after, inq_after, cmp_after, tau_after, cond_after, obj_after, norm_after = \
                progress_summary_df_clean(v2, v3, v4, tau, G, x, Q, c, ld1,
                                        norm_before, pr_before, inq_before, cmp_before,
                                        tau_before, cond_before, obj_before)

        display(summary_df)
        summary_df.to_excel(f"progress_summary_{mat_files}.xlsx", index=False)

        print("Condition number (1-norm):", cond_after)
        print("Objective after:", obj_after)
        print(f"k={k}")
        print(f"k_red={k_red}")

In [14]:
highlighted_df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470,471,472,473,474,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,563,564,565,566,567,568,569,570,571,572,573,574,575,576,577,578,579,580,581,582,583,584,585,586,587,588,589,590,591,592,593,594,595,596,597,598,599,600,601,602,603,604,605,606,607,608,609,610,611,612,613,614,615,616,617,618,619,620,621,622,623,624,625,626,627,628,629,630,631,632,633,634,635,636,637,638,639,640,641,642,643,644,645,646,647,648,649,650,651,652,653,654,655,656,657,658,659,660,661,662,663,664,665,666,667,668,669,670,671,672,673,674,675,676,677,678,679,680,681,682,683,684,685,686,687,688,689,690,691,692,693,694,695,696,697,698,699,700,701,702,703,704,705,706,707,708,709,710,711,712,713,714,715,716,717,718,719,720,721,722,723,724,725,726,727,728,729,730,731,732,733,734,735,736,737,738,739,740,741,742,743,744,745,746,747,748,749,750,751,752,753,754,755,756,757,758,759,760,761,762,763,764,765,766,767,768,769,770,771,772,773,774,775,776,777,778,779,780,781,782,783,784,785,786,787,788,789,790,791,792,793,794,795,796,797,798,799,800,801,802,803,804,805,806,807,808,809,810,811,812,813,814,815,816,817,818,819,820,821,822,823,824,825,826,827,828,829,830,831,832,833,834,835,836,837,838,839,840,841,842,843,844,845,846,847,848,849,850,851,852,853,854,855,856,857,858,859,860,861,862,863,864,865,866,867,868,869,870,871,872,873,874,875,876,877,878,879,880,881,882,883,884,885,886,887,888,889,890,891,892,893,894,895,896,897,898,899,900,901,902,903,904,905,906,907,908,909,910,911,912,913,914,915,916,917,918,919,920,921,922,923,924,925,926,927,928,929,930,931,932,933,934,935,936,937,938,939,940,941,942,943,944,945,946,947,948,949,950,951,952,953,954,955,956,957,958,959,960,961,962,963,964,965,966,967,968,969,970,971,972,973,974,975,976,977,978,979,980,981,982,983,984,985,986,987,988,989,990,991,992,993,994,995,996,997,998,999,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021